# 03 — İngilizce Ad Köprüsü ve DDInter Eşlemesi (İP3 - Kişi A Bölümü)

**Sorumlu:** Kişi A  
**Rol:** Türk İlaç Verisi (TİTCK + SKRS) Temizliği, Bileşen Ayrıştırması & İngilizce Ad Köprüsü  
**Girdi Dosyaları:**
- `data/interim/urun.csv` (Adım 1 çıktısı)
- `data/interim/urun_etken.csv` (Adım 2 çıktısı)
- `data/raw/skrs_2026-10-14.xlsx` (SKRS Aktif ve Pasif sekmeleri)
- `data/raw/ddinter_code_*.csv` (8 adet DDInter etkileşim dosyası)

**Çıktı Dosyaları:**
- `data/interim/etken_madde.csv` (Yol A, Yol B ve SKRS eşleşmeleri tamamlanmış, urun_sayisi hesaplanmış taslak — Gün 5 başında Kişi B'ye devredilir)
- `data/interim/ddinter_ilac.csv` (Adım 4'ün ilk maddesi: 1.939 benzersiz ilaç listesi)

---

### Veri Birleştirme Planı Sabit Kuralları ve Prensipleri:
1. **ATC Adı Sözlüğü (Adım 3):** SKRS'in tüm satırlarından (aktif + pasif) `ATC kodu → ATC adı` sözlüğü çıkarılır. ATC adı ürünün kendi SKRS satırından değil, bu sözlükten alınır; böylece SKRS'te satırı olmayan ruhsatlı ürünler de Yol A'yı kullanabilir. Birden fazla ad geçen kodlar için en sık geçen (mode) ad alınır.
2. **Yol A (Tek Bileşenli Ürünler):** Tek bileşenli, 7 haneli ATC'li, `atc_celiski=0` olan ürünlerde ATC adı doğrudan İngilizce karşılık olarak kullanılır (`deksketoprofen` → `dexketoprofen`). ATC adında `combinations`, `excl.` veya `various` geçenler elenir.
3. **Yol B (Kombine Ürünler):** Kombine ürünlerde ATC adı `and` ve `,` ile bölünür. Parça sayısı bileşen sayısına eşitse `rapidfuzz` ile eşleştirme yapılır (benzerlik $\ge 80$). `combinations`, `excl.` veya `various` içeren adlar bölünmez; Yol C ve D'ye bırakılır.
4. **Sabit Kural 5 (SKRS Ürünleri):** SKRS kaynaklı ürünlerin etkin maddesi zaten İngilizce ATC adı olduğundan Türkçe temizlikten geçmez, bileşenleri doğrudan `ad_en` olarak kaydedilir (`yol='SKRS'`).
5. **Adım 4 İlk Madde (ddinter_ilac.csv):** 8 DDInter dosyasından `DDInterID_A, Drug_A` ve `DDInterID_B, Drug_B` alt alta eklenip tekrarlar atılır (~1.939 satır) ve `ad_normalize()` uygulanır.
6. **Teslim Noktası:** Yol A, Yol B ve SKRS ile çözülen bileşenler belirlenip `urun_sayisi` azalan sırada hesaplanarak `etken_madde.csv` taslağı Kişi B'ye (Yol C ve Yol D için) devredilir.


In [1]:
import sys
import glob
from pathlib import Path
import pandas as pd
import numpy as np
from rapidfuzz import fuzz

# Proje kök dizinini ekle
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(kok))

from src.normalize import ad_normalize, eslesme_anahtari
from src.kopru import (
    skrs_atc_sozlugu_olustur,
    ddinter_ilac_tablosu_olustur,
    yol_a_esle,
    yol_b_esle,
    skrs_esle,
    etken_madde_olustur
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

print("✓ Ortam ve kütüphaneler başarıyla yüklendi.")
print(f"  Proje kökü: {kok}")


✓ Ortam ve kütüphaneler başarıyla yüklendi.
  Proje kökü: /Users/mervesudeyanar/Desktop/ddi-final


In [2]:
# Ara veri dosyalarının yüklenmesi
urun_yol = kok / 'data' / 'interim' / 'urun.csv'
ue_yol = kok / 'data' / 'interim' / 'urun_etken.csv'
skrs_yol = kok / 'data' / 'raw' / 'skrs_2026-10-14.xlsx'
ddinter_yollar = sorted(glob.glob(str(kok / 'data' / 'raw' / 'ddinter_code_*.csv')))

df_urun = pd.read_csv(urun_yol, dtype=str)
df_urun['atc_celiski'] = pd.to_numeric(df_urun['atc_celiski'], errors='coerce').fillna(0).astype(int)
df_urun['kapsam_disi'] = pd.to_numeric(df_urun['kapsam_disi'], errors='coerce').fillna(0).astype(int)
df_urun['barkod_gecersiz'] = pd.to_numeric(df_urun['barkod_gecersiz'], errors='coerce').fillna(0).astype(int)

df_ue = pd.read_csv(ue_yol, dtype=str)
df_ue['sira'] = pd.to_numeric(df_ue['sira'], errors='coerce').fillna(1).astype(int)

print("✓ Veri setleri yüklendi:")
print(f"  urun.csv satır sayısı         : {len(df_urun):,}")
print(f"  urun_etken.csv satır sayısı   : {len(df_ue):,}")
print(f"  Benzersiz Türkçe bileşen sayısı: {df_ue['bilesen_tr'].nunique():,}")
print(f"  Bulunan DDInter dosyası sayısı: {len(ddinter_yollar)}")


✓ Veri setleri yüklendi:
  urun.csv satır sayısı         : 23,801
  urun_etken.csv satır sayısı   : 32,204
  Benzersiz Türkçe bileşen sayısı: 3,111
  Bulunan DDInter dosyası sayısı: 8


## 1. SKRS ATC Adı Sözlüğünün Oluşturulması

SKRS'in tüm satırlarından (aktif + pasif) `ATC kodu → ATC adı` sözlüğü çıkarılır.  
- Birden fazla ad geçen ATC kodları için en sık geçen ad (`mode()`) temel alınır.
- Resmi SKRS ATC adı olduğu gibi korunur; keyfi parantez temizliği yapılmaz, şüpheli durumlar `docs/acik_sorular.md` dosyasına kaydedilir.


In [3]:
atc_sozlugu = skrs_atc_sozlugu_olustur(skrs_yol)

print(f"✓ SKRS ATC Sözlüğü oluşturuldu:")
print(f"  Sözlükteki benzersiz ATC kodu sayısı: {len(atc_sozlugu):,}")

# Örnek sözlük kayıtları
ornek_kodlar = ['M01AE17', 'B01AA03', 'J01FA09', 'C09DB01', 'A10BK03', 'N02BE01', 'A11GA01']
print()
print("Örnek Sözlük Eşleşmeleri:")
for kod in ornek_kodlar:
    print(f"  {kod:<8} -> {atc_sozlugu.get(kod, 'BULUNAMADI')}")


✓ SKRS ATC Sözlüğü oluşturuldu:
  Sözlükteki benzersiz ATC kodu sayısı: 1,795

Örnek Sözlük Eşleşmeleri:
  M01AE17  -> dexketoprofen
  B01AA03  -> warfarin
  J01FA09  -> clarithromycin
  C09DB01  -> valsartan and amlodipine
  A10BK03  -> empagliflozin
  N02BE01  -> paracetamol
  A11GA01  -> ascorbic acid (vit c)


## 2. DDInter İlaç Tablosunun Çıkarılması (`ddinter_ilac.csv`)

8 DDInter CSV dosyasından (`ddinter_code_A.csv` ... `V.csv`):
- `(DDInterID_A, Drug_A)` ve `(DDInterID_B, Drug_B)` alt alta eklenip tekrarlar atılır.
- `ad_normalize()` ile `ad_norm` sütunu oluşturulur.
- Çıktı: `data/interim/ddinter_ilac.csv` (yaklaşık 1.939 satır).


In [4]:
df_ddinter_ilac = ddinter_ilac_tablosu_olustur(ddinter_yollar)

ddinter_cikti_yol = kok / 'data' / 'interim' / 'ddinter_ilac.csv'
df_ddinter_ilac.to_csv(ddinter_cikti_yol, index=False)

print(f"✓ ddinter_ilac.csv başarıyla kaydedildi: {ddinter_cikti_yol}")
print(f"  Satır sayısı: {len(df_ddinter_ilac):,}")
print(f"  Benzersiz ddinter_id sayısı: {df_ddinter_ilac['ddinter_id'].nunique():,}")
print(f"  Benzersiz ad_norm sayısı   : {df_ddinter_ilac['ad_norm'].nunique():,}")

print()
print("İlk 5 DDInter İlaç Kaydı:")
print(df_ddinter_ilac.head())


✓ ddinter_ilac.csv başarıyla kaydedildi: /Users/mervesudeyanar/Desktop/ddi-final/data/interim/ddinter_ilac.csv
  Satır sayısı: 1,939
  Benzersiz ddinter_id sayısı: 1,939
  Benzersiz ad_norm sayısı   : 1,939

İlk 5 DDInter İlaç Kaydı:
    ddinter_id        drug_name          ad_norm
0     DDInter1         Abacavir         abacavir
1    DDInter10      Acamprosate      acamprosate
2   DDInter100  Anthrax vaccine  anthrax vaccine
3  DDInter1000       Ivosidenib       ivosidenib
4  DDInter1001      Ixabepilone      ixabepilone


## 3. Adım 3 — Yol A: Tek Bileşenli Ürünler

**Kural:**
- Tek bileşenli ürün (`bilesen_sayisi == 1`), 7 haneli ATC kodu, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adı SKRS sözlüğünden doğrudan İngilizce karşılık olarak alınır (`deksketoprofen` → `dexketoprofen`).
- Genel kombinasyon ifadeleri (`combinations`, `excl.`, `various`) içeren adlar elenir.
- Bir bileşen birden fazla tek bileşenli üründe geçiyorsa en çok oy alan (mode) ATC/ad çifti seçilir.


In [5]:
df_yol_a = yol_a_esle(df_urun, df_ue, atc_sozlugu)

print(f"✓ Yol A eşleştirmesi tamamlandı:")
print(f"  Yol A ile eşleşen benzersiz bileşen sayısı: {len(df_yol_a):,}")

print()
print("Yol A Örnek Eşleşmeler:")
print(df_yol_a[['bilesen_tr', 'ad_en', 'atc']].head(10))


✓ Yol A eşleştirmesi tamamlandı:
  Yol A ile eşleşen benzersiz bileşen sayısı: 1,922

Yol A Örnek Eşleşmeler:
                                          bilesen_tr                                              ad_en      atc
0                            % 20'lik insan albumini                                            albumin  B05AA01
1  % 25 insulin lispro cozelti, % 75 protamin sus...                                     insulin lispro  A10AD04
2  % 25 insulin lispro, % 75 insulin lispro protamin                                     insulin lispro  A10AD04
3                            % 25'lik insan albumini                                            albumin  B05AA01
4  % 50 insulin lispro cozelti, % 50 protamin sus...                                     insulin lispro  A10AD04
5  % 50 insulin lispro, % 50 insulin lispro protamin                                     insulin lispro  A10AD04
6   %25 insulin lispro / %75 insulin lispro protamin                                     insulin li

## 4. Adım 3 — Yol B: Kombine Ürünler

**Kural:**
- Kombine ürünler (`bilesen_sayisi > 1`), 7 haneli ATC, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adında `combinations`, `excl.` veya `various` varsa bölünmez (Yol C ve D'ye bırakılır).
- ATC adı `and` ve `,` ile bölünür. Parça sayısı bileşen sayısına eşitse `rapidfuzz` ile eşleştirme yapılır (benzerlik skoru $\ge 80$).
- Yol A'da eşleşen bileşenler mükerrer eşlemeyi önlemek için hariç tutulur.


In [6]:
eslesenler_a = set(df_yol_a['bilesen_tr'])
df_yol_b = yol_b_esle(df_urun, df_ue, atc_sozlugu, onceki_eslesmeler=eslesenler_a)

print(f"✓ Yol B eşleştirmesi tamamlandı:")
print(f"  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: {len(df_yol_b):,}")

print()
print("Yol B Örnek Eşleşmeler:")
print(df_yol_b[['bilesen_tr', 'ad_en', 'atc', 'eslesme_skoru']].head(10))


✓ Yol B eşleştirmesi tamamlandı:
  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: 66

Yol B Örnek Eşleşmeler:
             bilesen_tr                  ad_en      atc  eslesme_skoru
0            sakubitril             sacubitril  C09DX04      90.000000
1            trastuzuma            trastuzumab  L01FY01      95.238095
2       etinilestradiol       ethinylestradiol  G03AA12      90.322581
3           fluticasone            fluticasone  R03AK06     100.000000
4           biktegravir            bictegravir  J05AR20      90.909091
5          emtrisitabin          emtricitabine  J05AR20      88.000000
6  tenofovir alefenamid  tenofovir alafenamide  J05AR20      92.682927
7             lopinavir              lopinavir  J05AR10     100.000000
8           velpatasvir            velpatasvir  J05AP56     100.000000
9         voksilaprevir           voxilaprevir  J05AP56      88.000000


## 5. Sabit Kural 5 — SKRS Kaynaklı Ürünler (`yol='SKRS'`)

SKRS'te aktif olup ruhsatlıda olmayan ürünlerin etkin maddesi doğrudan SKRS İngilizce ATC Adından geldiği için, bileşenler doğrudan `ad_en` olarak alınır ve `yol='SKRS'` atanır.


In [7]:
eslesenler_ab = eslesenler_a | set(df_yol_b['bilesen_tr'])
df_skrs = skrs_esle(df_urun, df_ue, onceki_eslesmeler=eslesenler_ab)

print(f"✓ SKRS eşleştirmesi tamamlandı:")
print(f"  SKRS ile YENİ eşleşen benzersiz bileşen sayısı: {len(df_skrs):,}")

print()
print("SKRS Örnek Eşleşmeler:")
print(df_skrs[['bilesen_tr', 'ad_en', 'atc']].head(10))


✓ SKRS eşleştirmesi tamamlandı:
  SKRS ile YENİ eşleşen benzersiz bileşen sayısı: 90

SKRS Örnek Eşleşmeler:
                                          bilesen_tr                                              ad_en      atc
0           other non therapeutic auxiliary products           other non therapeutic auxiliary products    V07AY
1  fat/carbohydrates/proteins/minerals/vitamins, ...  fat/carbohydrates/proteins/minerals/vitamins, ...    V06DB
2                                        paracetamol                                        paracetamol  N02BE01
3                                        palbociclib                                        palbociclib  L01EF01
4                                  allergen extracts                                  allergen extracts    V01AA
5                               peritoneal dialytics                               peritoneal dialytics     B05D
6                                    sodium chloride                                    sodium chlor

## 6. `etken_madde.csv` Taslağının Oluşturulması ve Kaydedilmesi

Tüm 3.111 benzersiz Türkçe bileşen için:
- `urun_sayisi`: Bu bileşeni içeren benzersiz barkod / ürün sayısı.
- Yol A, Yol B ve SKRS eşleşmeleri birleştirilir.
- Eşleşmeyenler `yol='eslesmedi'` olarak Kişi B'ye (Yol C ve Yol D için) devredilir.
- Tablo `urun_sayisi` azalan sırada sıralanarak kaydedilir.


In [8]:
df_etken_madde = etken_madde_olustur(df_urun, df_ue, atc_sozlugu)

em_cikti_yol = kok / 'data' / 'interim' / 'etken_madde.csv'
df_etken_madde.to_csv(em_cikti_yol, index=False)

print(f"✓ etken_madde.csv başarıyla kaydedildi: {em_cikti_yol}")
print(f"  Toplam benzersiz bileşen sayısı: {len(df_etken_madde):,}")
print()
print("Yol Dağılımı:")
print(df_etken_madde['yol'].value_counts())

print()
print("En Sık Kullanılan İlk 15 Bileşen:")
print(df_etken_madde.head(15))


✓ etken_madde.csv başarıyla kaydedildi: /Users/mervesudeyanar/Desktop/ddi-final/data/interim/etken_madde.csv
  Toplam benzersiz bileşen sayısı: 3,111

Yol Dağılımı:
yol
A            1922
eslesmedi    1033
SKRS           90
B              66
Name: count, dtype: int64

En Sık Kullanılan İlk 15 Bileşen:
                                  bilesen_tr                                     ad_en      atc        yol  urun_sayisi
0                              sodyum klorur                           sodium chloride  B05XA03          A         1059
1                                   dekstroz                             carbohydrates  B05BA03          A          720
2                            kalsiyum klorur                                       NaN      NaN  eslesmedi          400
3                            potasyum klorur                        potassium chloride  B05XA01          A          387
4                          hidroklorotiyazid                                       NaN      NaN  e

## 7. DDInter Tam Ad Eşleşme Ön Testi

Yol A, Yol B ve SKRS ile üretilen İngilizce adların (`ad_en`) DDInter veritabanında (`ddinter_ilac.csv`) birebir tam eşleşme durumunun kontrolü.


In [9]:
ddinter_dict = df_ddinter_ilac.set_index('ad_norm')['ddinter_id'].to_dict()

# Eşleşenlerin DDInter karşılığı
df_test = df_etken_madde.copy()
df_test['ddinter_id'] = df_test['ad_en'].map(ddinter_dict)

print("DDInter Tam Ad Eşleşme Durumu (Yol Bazında):")
for y in ['A', 'B', 'SKRS', 'eslesmedi']:
    sub = df_test[df_test['yol'] == y]
    tam_eslesen = sub['ddinter_id'].notna().sum()
    toplam = len(sub)
    oran = (tam_eslesen / toplam * 100) if toplam > 0 else 0
    print(f"  Yol {y:<9}: Toplam = {toplam:>4}, DDInter'da Tam Eşleşen = {tam_eslesen:>4} (%{oran:5.1f})")

toplam_tam_eslesen = df_test['ddinter_id'].notna().sum()
print()
print(f"Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: {toplam_tam_eslesen:,} / {len(df_test):,} (%{toplam_tam_eslesen/len(df_test)*100:.1f})")


DDInter Tam Ad Eşleşme Durumu (Yol Bazında):
  Yol A        : Toplam = 1922, DDInter'da Tam Eşleşen = 1197 (% 62.3)
  Yol B        : Toplam =   66, DDInter'da Tam Eşleşen =   32 (% 48.5)
  Yol SKRS     : Toplam =   90, DDInter'da Tam Eşleşen =   31 (% 34.4)
  Yol eslesmedi: Toplam = 1033, DDInter'da Tam Eşleşen =    0 (%  0.0)

Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: 1,260 / 3,111 (%40.5)


## 8. Doğrulama ve Teslim Kontrol Listesi

Kişi A sorumluluk alanındaki teslimatların doğrulanması:
- [x] SKRS ATC Adı sözlüğü doğru ve tutarlı oluşturuldu.
- [x] `ddinter_ilac.csv` yaklaşık 1.939 satır olarak çıkarıldı.
- [x] Yol A (tek bileşenli) ve Yol B (kombine) kurallara tam sadık kalınarak uygulandı.
- [x] SKRS ürünleri Kural 5 uyarınca `yol='SKRS'` olarak tamamlandı.
- [x] `data/interim/etken_madde.csv` taslağı oluşturuldu ve Kişi B'ye (Yol C, Yol D ve PubChem eşleşmeleri için) devredilmeye hazır.


In [10]:
assert 1900 <= len(df_ddinter_ilac) <= 2000, f"HATA: ddinter_ilac satır sayısı (~1.939) aralık dışında: {len(df_ddinter_ilac)}!"
assert len(df_etken_madde) == 3111, f"HATA: etken_madde satır sayısı 3111 olmalı, {len(df_etken_madde)} çıktı!"
assert (df_etken_madde['yol'] == 'A').sum() > 1800, "HATA: Yol A sayısı beklenenden düşük!"
assert (df_etken_madde['yol'] == 'B').sum() > 50, "HATA: Yol B sayısı beklenenden düşük!"
assert (df_etken_madde['yol'] == 'SKRS').sum() > 80, "HATA: SKRS sayısı beklenenden düşük!"

print("════════════════════════════════════════════════════════════════")
print("✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!")
print("✓ Kişi A teslimatı tamamlandı. Dosyalar Kişi B'ye devredilebilir.")
print("════════════════════════════════════════════════════════════════")


════════════════════════════════════════════════════════════════
✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!
✓ Kişi A teslimatı tamamlandı. Dosyalar Kişi B'ye devredilebilir.
════════════════════════════════════════════════════════════════
